# DESeq2: Basic Differential Expression (DE) analysis

## Objective: Carry out a basic set of DE interaction analysis using DESeq2 and visualize the results

## The objective is to identify genes whose differential effect due to treatment depends on the genotype 

### Load packages

In [ ]:
library(tidyverse)
library(DESeq2)
library(dendextend)
library(RColorBrewer)

### Load the 2019 pilot dds object from image file

In [ ]:
curdir <- "/home/jovyan/work/scratch/analysis_output"
imgdir <- file.path(curdir, "img")

imgfile <- file.path(imgdir, "pilotdds2019.RData")

imgfile

attach(imgfile)

tools::md5sum(imgfile)

### List the objects that have been attached
ls(2)

dds2019 <- dds2019

detach(pos = 2)

## Before carrying out an interaction analysis, let's review the steps for conducting a DE analysis

### Inspect the dds2019 object

In [ ]:
dds2019

### Note that the design is additive 

In [ ]:
design(dds2019)

### The steps for a basic analysis are: estimate size factors, estimate dispersion parameters, and then carry out DE analysis

In [ ]:
### First copy dds2019 to ddsadd
ddsadd <- dds2019
### Estimate Size Factors
ddsadd <- estimateSizeFactors(ddsadd)
### Estimate Dispersion parameters (for each gene)
ddsadd <- estimateDispersions(ddsadd)
### Fit NB MLE model
ddsadd <- DESeq(ddsadd)
### Rlog "normalized" expressions
#rldadd <- rlog(ddsadd)

### Identify differentially expressed genes with respect to condition using pH4 as reference

In [ ]:
results(ddsadd, contrast = c("condition", "pH8", "pH4"), tidy = TRUE) %>%
    arrange(desc(-padj)) %>% 
        head(5)

In [ ]:
## Interaction analysis

In [ ]:
### The first step is to specify the design

In [ ]:
### Additive design
ddsmult <- dds2019
design(ddsmult)

Update the design by adding the term genotype:condition

In [ ]:
design(ddsmult) <- formula(~ genotype + condition + genotype:condition)
design(ddsmult)

In [ ]:
### Now repeat the steps: estimate size factors, estimate dispersion followed by the analysis

In [ ]:
### Estimate Size Factors
ddsmult <- estimateSizeFactors(ddsmult)
### Estimate Dispersion parameters (for each gene)
ddsmult <- estimateDispersions(ddsmult)
### Fit NB MLE model
ddsmultres <- DESeq(ddsmult)
### Rlog "normalized" expressions
#rldmult <- rlog(ddsmult, blind = TRUE)

Look at the results (compare the first line to that of the DE analysis

In [ ]:
results(ddsmultres)

List the top 5 hits with respect to adjusted P-value

In [ ]:
results(ddsmultres, tidy = TRUE) %>%
    arrange(desc(-padj)) %>%
        head(5)

Visualize the results

In [ ]:
### Merge gene expression with meta data
myDEplotData <- function(mydds, geneid, mergelab) {
    counts(mydds, normalize = TRUE) %>%
        as_tibble(rownames="gene") %>%
        filter(gene == geneid) %>%
        gather(Label, geneexp, -gene) %>%
        select(-gene) -> genedat

    colData(mydds) %>%
        as.data.frame %>%
        as_tibble %>%
        full_join(genedat, by = mergelab) -> genedat
    
    return(genedat)
}


### Alow for coloring with respect to another factor
myDEplot <- function(mydds, geneid, grpvar, colvar, mergelab) {
    mydat <- myDEplotData(mydds, geneid, mergelab)
    ggplot(mydat, aes_string(x=grpvar, y = "geneexp", col = colvar))+ geom_point()
}

Compare the DE effect of condition within WT compares to sre1d. This suggests that the DE with respect to condition depends on genotype. 

In [ ]:
myDEplot(ddsmult, "CNAG_03518", "genotype", "condition", "Label")

##Exercise: Compare the size factors and dispersion estimates between ddsmult and ddsad

In [ ]:
sessionInfo()